In [0]:
%pip install python-dotenv --quiet
dbutils.library.restartPython()

In [0]:
import os
from dotenv import load_dotenv

load_dotenv("/Workspace/Users/ik.kukoo@gmail.com/.env", override=True)

STORAGE_ACCOUNT = "internshipdatalake"
CONTAINER       = "raw"
CONTAINER_PATH  = "batch-data"
TABLE           = "food_fornecedores.csv"

adls_options = {
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_ID"),
    f"fs.azure.account.oauth2.client.secret.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        os.getenv("ADLS_CLIENT_SECRET"),
    f"fs.azure.account.oauth2.client.endpoint.{STORAGE_ACCOUNT}.dfs.core.windows.net":
        f"https://login.microsoftonline.com/{os.getenv('ADLS_TENANT_ID')}/oauth2/token",
}

PATH_FORNECEDORES = f"abfss://{CONTAINER}@{STORAGE_ACCOUNT}.dfs.core.windows.net/{CONTAINER_PATH}/{TABLE}"

print("✅ Config OK →", PATH_FORNECEDORES)

In [0]:
# Verificando o conteúdo da Tabela food_fornecedores.csv
df = (
    spark.read
    .options(**adls_options)
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(PATH_FORNECEDORES)
)

print(f"Linhas: {df.count()} | Colunas: {len(df.columns)}")
df.printSchema()

In [0]:
# Amostra dos dados
df.show(10, truncate=False)

In [0]:
# Estatísticas descritivas
df.describe().show(truncate=False)

In [0]:
# Análise de nulos
from pyspark.sql.functions import col, sum as spark_sum

nulls = df.select([
    spark_sum(col(c).isNull().cast("int")).alias(c)
    for c in df.columns
])
print("🔍 Nulos por coluna:")
nulls.show(truncate=False)

In [0]:
# Fornecedores ativos vs inativos
df.groupBy("is_ativo").count().show()

In [0]:
# Distribuição por categoria fornecida
df.groupBy("categoria_fornecida").count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
# Distribuição por tipo de fornecedor
df.groupBy("tipo_fornecedor").count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
# Distribuição por UF de origem
df.groupBy("uf_origem").count().orderBy("count", ascending=False).show(truncate=False)

In [0]:
# Lead time por tipo de fornecedor
from pyspark.sql.functions import avg, min as spark_min, max as spark_max

df.groupBy("tipo_fornecedor").agg(
    avg("lead_time_dias").alias("media_lead_time"),
    spark_min("lead_time_dias").alias("min_lead_time"),
    spark_max("lead_time_dias").alias("max_lead_time")
).orderBy("media_lead_time", ascending=False).show(truncate=False)

In [0]:
# Certificações mais comuns
from pyspark.sql.functions import explode, split

df.select(explode(split(col("certificacoes"), ";")).alias("certificacao")) \
  .groupBy("certificacao").count() \
  .orderBy("count", ascending=False) \
  .show(truncate=False)